## Importing the necessary libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

## 1) Heart disease

In [2]:
# Column names 
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

# 1. Load dataset
df1 = pd.read_csv("processed.cleveland.data", header=None, names=columns)

# 2. Replace missing values ('?') with NaN
df1 = df1.replace("?", np.nan)

# 3. Convert all columns to numeric
df1 = df1.apply(pd.to_numeric, errors="coerce")

# 4. Impute missing values with median
df1 = df1.fillna(df1.median())

# 5. Encode categorical variables
categorical_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
for col in categorical_cols:
    le = LabelEncoder()
    df1[col] = le.fit_transform(df1[col].astype(str))

# 6. Define features and target
X1 = df1.drop(columns=["num"])
y1 = df1["num"]

# Convert to binary target (0 = no disease, 1 = disease)
y1 = np.where(y1 > 0, 1, 0)

print("Dataset 1 (Cleveland) shape:", X1.shape)
print("Class distribution:", np.bincount(y1))

# 7. Scale features
scaler = StandardScaler()
X1_scaled = scaler.fit_transform(X1)

# 8. Train-test split
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X1_scaled, y1, test_size=0.2, random_state=42, stratify=y1
)

# 9. Combine into train/test DataFrames
train1 = pd.DataFrame(X_train1, columns=X1.columns)
train1["target"] = y_train1

test1 = pd.DataFrame(X_test1, columns=X1.columns)
test1["target"] = y_test1

# 10. Print summary
print("\nTrain1 shape:", train1.shape)
print("Test1 shape:", test1.shape)
print("Target meaning: 0 = No Heart Disease, 1 = Heart Disease")

# Preview first rows
print(train1.head())


Dataset 1 (Cleveland) shape: (303, 13)
Class distribution: [164 139]

Train1 shape: (242, 14)
Test1 shape: (61, 14)
Target meaning: 0 = No Heart Disease, 1 = Heart Disease
        age       sex        cp  trestbps      chol       fbs   restecg  \
0 -0.713556  0.686202  0.877985 -0.437648  0.528268 -0.417635  1.016684   
1  0.062176  0.686202 -1.208521 -0.096170  0.296121 -0.417635 -0.996749   
2 -0.048643 -1.457296 -1.208521  0.017656  0.799106  2.394438  1.016684   
3 -0.048643  0.686202 -1.208521 -1.348256  1.205363 -0.417635 -0.996749   
4  0.283813 -1.457296  0.877985  0.472960 -0.110136 -0.417635 -0.996749   

    thalach     exang   oldpeak     slope        ca      thal  target  
0  0.717808 -0.696631 -0.465514  0.649113 -0.711131  1.223208       1  
1  0.236138 -0.696631 -0.896862 -0.976352 -0.711131 -0.870759       0  
2  0.411291  1.435481 -0.896862 -0.976352  0.360873 -0.870759       0  
3  0.279926 -0.696631 -0.896862 -0.976352 -0.711131  1.223208       0  
4 -1.165083  1.43

## 2) statlog heart

In [3]:
# Column names 
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

# 1. Load dataset (space-separated)
df2 = pd.read_csv("heart.dat", sep=" ", header=None, names=columns)

# 2. Handle missing values (if any)
for col in df2.columns:
    if df2[col].isnull().sum() > 0:
        if df2[col].dtype == "object":
            df2[col] = df2[col].fillna(df2[col].mode()[0])
        else:
            df2[col] = df2[col].fillna(df2[col].median())

# 3. Encode categorical variables
categorical_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
for col in categorical_cols:
    le = LabelEncoder()
    df2[col] = le.fit_transform(df2[col].astype(str))

# 4. Define features and target
X2 = df2.drop(columns=["target"])
y2 = df2["target"]

# Convert target: in Statlog, 1 = disease, 2 = no disease → fix to 0/1
y2 = np.where(y2 == 2, 0, 1)

print("Dataset 2 (Statlog) shape:", X2.shape)
print("Class distribution:", np.bincount(y2))

# 5. Scale features
scaler = StandardScaler()
X2_scaled = scaler.fit_transform(X2)

# 6. Train-test split (stratify to keep balance)
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2_scaled, y2, test_size=0.2, random_state=42, stratify=y2
)

# 7. Combine into train/test DataFrames
train2 = pd.DataFrame(X_train2, columns=X2.columns)
train2["target"] = y_train2

test2 = pd.DataFrame(X_test2, columns=X2.columns)
test2["target"] = y_test2

# 8. Print summary
print("\nTrain2 shape:", train2.shape)
print("Test2 shape:", test2.shape)
print("Target meaning: 0 = No Heart Disease, 1 = Heart Disease")

# Preview first rows
print("\nHead of training set:")
print(train2.head())


Dataset 2 (Statlog) shape: (270, 13)
Class distribution: [120 150]

Train2 shape: (216, 14)
Test2 shape: (54, 14)
Target meaning: 0 = No Heart Disease, 1 = Heart Disease

Head of training set:
        age     sex        cp  trestbps      chol       fbs   restecg  \
0  1.602109  0.6895 -0.183559   0.48549  0.084138 -0.417029  0.981664   
1  0.062325  0.6895 -1.238045  -0.07541  0.239206 -0.417029 -1.026285   
2 -2.797275  0.6895 -1.238045  -0.07541 -0.885033 -0.417029  0.981664   
3  0.502263  0.6895 -0.183559  -0.29977 -0.613665  2.397916 -1.026285   
4 -2.137367  0.6895  0.870928  -0.63631 -1.001334 -0.417029 -1.026285   

    thalach     exang   oldpeak     slope        ca      thal  target  
0 -0.159054 -0.701222  0.831083  0.676419  2.472682  1.230232       0  
1  0.230172 -0.701222 -0.918565 -0.954234 -0.711535 -0.858841       1  
2  2.262800 -0.701222 -0.918565 -0.954234 -0.711535 -0.858841       1  
3 -0.678023 -0.701222  1.006048  0.676419  0.349871  0.185695       0  
4 -0.851

## 3) spect heart

In [4]:
# 1. Load train and test
train_raw = pd.read_csv("SPECT.train", header=None)
test_raw  = pd.read_csv("SPECT.test", header=None)

print("Train shape before renaming:", train_raw.shape)
print("Test shape before renaming:", test_raw.shape)

# 2. Assign column names
feature_cols = [f"F{i}" for i in range(1, 23)]  # 22 features
columns = ["target"] + feature_cols
train_raw.columns = columns
test_raw.columns = columns

# 3. Merge train and test
df3 = pd.concat([train_raw, test_raw], axis=0).reset_index(drop=True)
print("Combined dataset shape:", df3.shape)

# 4. Handle missing values
num_cols = df3.select_dtypes(include=["number"]).columns.drop("target", errors="ignore")
cat_cols = df3.select_dtypes(exclude=["number"]).columns

if df3[num_cols].isnull().sum().sum() > 0:
    df3[num_cols] = df3[num_cols].fillna(df3[num_cols].median())

for col in cat_cols:
    if df3[col].isnull().sum() > 0:
        df3[col] = df3[col].fillna(df3[col].mode()[0])

# 5. Normalize numeric features
scaler = MinMaxScaler()
df3[num_cols] = scaler.fit_transform(df3[num_cols])

# 6. Encode target (SPECT: '1' = abnormal, '0' = normal)
le = LabelEncoder()
df3["target"] = le.fit_transform(df3["target"])

print("\nClass distribution (whole dataset):")
print(df3["target"].value_counts())

# 7. Train-test split
X3 = df3.drop("target", axis=1)
y3 = df3["target"]

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)

# 8. Combine into train/test DataFrames
train3 = pd.DataFrame(X_train3, columns=X3.columns)
train3["target"] = y_train3

test3 = pd.DataFrame(X_test3, columns=X3.columns)
test3["target"] = y_test3

# 9. Print summary
print("\nTrain3 shape:", train3.shape)
print("Test3 shape:", test3.shape)
print("Target meaning: 0 = Normal, 1 = Abnormal (Heart disease)")

# Preview first rows
print("\nHead of training set:")
print(train3.head())


Train shape before renaming: (80, 23)
Test shape before renaming: (187, 23)
Combined dataset shape: (267, 23)

Class distribution (whole dataset):
target
1    212
0     55
Name: count, dtype: int64

Train3 shape: (213, 23)
Test3 shape: (54, 23)
Target meaning: 0 = Normal, 1 = Abnormal (Heart disease)

Head of training set:
      F1   F2   F3   F4   F5   F6   F7   F8   F9  F10  ...  F14  F15  F16  \
147  1.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  ...  0.0  0.0  1.0   
264  1.0  0.0  1.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  ...  0.0  1.0  1.0   
19   1.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  ...  0.0  0.0  0.0   
149  1.0  1.0  1.0  0.0  1.0  0.0  1.0  1.0  0.0  1.0  ...  0.0  0.0  0.0   
244  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  1.0  0.0  1.0   

     F17  F18  F19  F20  F21  F22  target  
147  0.0  0.0  1.0  1.0  0.0  0.0       1  
264  0.0  0.0  0.0  0.0  0.0  0.0       0  
19   0.0  0.0  0.0  1.0  0.0  0.0       1  
149  0.0  1.0  1.0  0.0  1.0  1.0   

## 4) spectf heart

In [5]:
# 1. Load train & test
train = pd.read_csv("spectf.train", header=None)
test = pd.read_csv("spectf.test", header=None)

print("Train shape before renaming:", train.shape) 
print("Test shape before renaming:", test.shape)

# 2. Assign column names: 1 target + rest features
num_features = train.shape[1] - 1
columns = ["target"] + [f"feature_{i}" for i in range(1, num_features + 1)]
train.columns = columns
test.columns = columns

# 3. Merge train + test
df = pd.concat([train, test], axis=0).reset_index(drop=True)

# 4. Encode target (ensure 0/1 format)
if df["target"].dtype != "int64" and df["target"].dtype != "int32":
    le = LabelEncoder()
    df["target"] = le.fit_transform(df["target"])

# 5. Normalize numeric features
scaler = MinMaxScaler()
X = scaler.fit_transform(df.drop("target", axis=1))
y = df["target"]

# 6. Stratified train-test split
X_train4, X_test4, y_train4, y_test4 = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 7. Print target distributions
print("\nTrain target distribution:")
print(y_train4.value_counts())
print("\nTest target distribution:")
print(y_test4.value_counts())

print("\nTarget meaning for SPECTF dataset:")
print("0 → Normal heart diagnosis (no disease)")
print("1 → Abnormal heart diagnosis (disease present)")

# 8. Final ML-ready DataFrames
train4 = pd.DataFrame(X_train4, columns=columns[1:])
train4["target"] = y_train4.reset_index(drop=True)

test4 = pd.DataFrame(X_test4, columns=columns[1:])
test4["target"] = y_test4.reset_index(drop=True)

# Preview
print("\nHead of training set:")
print(train4.head())


Train shape before renaming: (80, 45)
Test shape before renaming: (187, 45)

Train target distribution:
target
1    169
0     44
Name: count, dtype: int64

Test target distribution:
target
1    43
0    11
Name: count, dtype: int64

Target meaning for SPECTF dataset:
0 → Normal heart diagnosis (no disease)
1 → Abnormal heart diagnosis (disease present)

Head of training set:
   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0   0.700000   0.483333   0.818182       0.76   0.672131   0.531250   
1   0.866667   0.866667   0.772727       0.86   0.770492   0.656250   
2   0.950000   0.883333   1.000000       0.86   0.868852   0.765625   
3   0.516667   0.516667   0.454545       0.42   0.672131   0.453125   
4   0.883333   0.933333   0.704545       0.72   0.704918   0.687500   

   feature_7  feature_8  feature_9  feature_10  ...  feature_36  feature_37  \
0   0.781250   0.730159   0.846154    0.589286  ...    0.718750    0.787234   
1   0.859375   0.841270   0.846154    

## 5) eeg eye state

In [6]:
from scipy.io import arff


# Load EEG Eye State (Eric) dataset
data_path = "EEG Eye State.arff"

# 1. Load ARFF file
data_arff = arff.loadarff(data_path)
df = pd.DataFrame(data_arff[0])

# 2. Decode target column if stored as bytes
if df['eyeDetection'].dtype == 'object' or str(df['eyeDetection'].dtype).startswith('|S'):
    df['eyeDetection'] = df['eyeDetection'].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

# 3. Rename columns: features + target
num_features = df.shape[1] - 1
df.columns = [f'feature_{i}' for i in range(1, num_features + 1)] + ['target']

# 4. Convert target to integer (0 = eyes open, 1 = eyes closed)
df['target'] = df['target'].astype(int)

# 5. Handle missing values using median
df = df.fillna(df.median(numeric_only=True))

# 6. Scale features to 0–1
scaler = MinMaxScaler()
df.iloc[:, :-1] = scaler.fit_transform(df.iloc[:, :-1])

# 7. Split into features (X) and target (y)
X = df.drop("target", axis=1)
y = df["target"]

# 8. Train–test split (80/20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 9. Combine features and target into final DataFrames
train_df = X_train.copy()
train_df['target'] = y_train

test_df = X_test.copy()
test_df['target'] = y_test


# Display dataset info
print("EEG Eye State dataset preprocessing complete.\n")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain target distribution:")
print(y_train.value_counts().rename("count").to_frame())

print("\nTest target distribution:")
print(y_test.value_counts().rename("count").to_frame())

print("\nTarget meaning:")
print("0 → Eyes open")
print("1 → Eyes closed")

print("\nSample of training set:")
print(train_df.head())

train5 = train_df
test5  = test_df

EEG Eye State dataset preprocessing complete.

Train shape: (11984, 15)
Test shape : (2996, 15)

Train target distribution:
        count
target       
0        6606
1        5378

Test target distribution:
        count
target       
0        1651
1        1345

Target meaning:
0 → Eyes open
1 → Eyes closed

Sample of training set:
       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
1123    0.010611   0.238787   0.551234   0.002613   0.514854   0.005028   
13517   0.010563   0.235695   0.551760   0.002609   0.512633   0.005134   
12368   0.010578   0.231776   0.550531   0.002600   0.513568   0.005171   
11320   0.010624   0.236726   0.554219   0.002603   0.516843   0.005167   
9923    0.010597   0.237653   0.553429   0.002581   0.511463   0.005144   

       feature_7  feature_8  feature_9  feature_10  feature_11  feature_12  \
1123    0.003535   0.014072   0.010705    0.495726    0.260331    0.426503   
13517   0.003488   0.015021   0.010746    0.495514    0.26

## 6) breast cancer wisconsin diagnostic

In [7]:
# 1. Load dataset
df = pd.read_csv("wdbc.data", header=None)

# 2. Column names
columns = [
    "ID", "diagnosis",
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
    "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se",
    "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst",
    "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]
df.columns = columns


# 3. Drop ID and encode target
df = df.drop("ID", axis=1)
df["target"] = df["diagnosis"].map({"M": 1, "B": 0})
df = df.drop("diagnosis", axis=1)


# 4. Handle missing values
df = df.fillna(df.median(numeric_only=True))
df["target"] = df["target"].fillna(0).astype(int)


# 5. Normalize features
feature_cols = df.columns[:-1]  # exclude target
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])


# 6. Train-test split (80/20 stratified)
X = df.drop("target", axis=1)
y = df["target"]

X_train6, X_test6, y_train6, y_test6 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 7. Reset indices to align features and target
X_train6 = X_train6.reset_index(drop=True)
X_test6  = X_test6.reset_index(drop=True)
y_train6 = y_train6.reset_index(drop=True)
y_test6  = y_test6.reset_index(drop=True)


# 8. Display dataset info
print("Full dataset shape:", X.shape)

print("\nTarget meaning for WDBC dataset:")
print("0 → Benign (no cancer)")
print("1 → Malignant (cancer present)")

print("\nTrain target distribution:")
print(y_train6.value_counts())

print("\nTest target distribution:")
print(y_test6.value_counts())

print("\nPreprocessed Train Data (first 5 rows):")
print(X_train6.head().assign(target=y_train6.head()))


# 9. Final train/test DataFrames
train6 = pd.concat([X_train6, y_train6], axis=1)
test6  = pd.concat([X_test6, y_test6], axis=1)


Full dataset shape: (569, 30)

Target meaning for WDBC dataset:
0 → Benign (no cancer)
1 → Malignant (cancer present)

Train target distribution:
target
0    285
1    170
Name: count, dtype: int64

Test target distribution:
target
0    72
1    42
Name: count, dtype: int64

Preprocessed Train Data (first 5 rows):
   radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
0     0.427801      0.457558        0.407090   0.277540         0.265686   
1     0.252686      0.090632        0.242278   0.135992         0.452920   
2     0.277770      0.394319        0.268399   0.157370         0.206554   
3     0.374793      0.433548        0.402944   0.229692         0.422858   
4     0.550381      0.356442        0.541151   0.403181         0.377088   

   compactness_mean  concavity_mean  concave_points_mean  symmetry_mean  \
0          0.145114        0.077296             0.165159       0.236364   
1          0.154684        0.093416             0.183897       0.454040   
2   

## 7) hepatitis

In [8]:
# 1. Load dataset
df = pd.read_csv("hepatitis.data", header=None, na_values="?")


# 2. Column names
columns = [
    "target", "age", "sex", "steroid", "antivirals", "fatigue", "malaise", "anorexia",
    "liver_big", "liver_firm", "spleen_palpable", "spiders", "ascites", "varices",
    "bilirubin", "alk_phosphate", "sgot", "albumin", "protime", "histology"
]
df.columns = columns


# 3. Convert target
# 1 = Die, 2 = Live → 1=Disease/Die, 0=Live/Normal
df["target"] = df["target"].map({2: 0, 1: 1})


# 4. Handle missing values
# Numeric columns → median
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical columns → mode
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Ensure target has no NaN
df["target"] = df["target"].fillna(0).astype(int)


# 5. Normalize numeric features (exclude target)
scaler = MinMaxScaler()
df[num_cols.drop("target")] = scaler.fit_transform(df[num_cols.drop("target")])


# 6. Train-test split (80/20, stratified)
X = df.drop("target", axis=1)
y = df["target"]

X_train7, X_test7, y_train7, y_test7 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reset indices
X_train7 = X_train7.reset_index(drop=True)
X_test7  = X_test7.reset_index(drop=True)
y_train7 = y_train7.reset_index(drop=True)
y_test7  = y_test7.reset_index(drop=True)

# 7. Print shapes and distributions before SMOTE
print("Train shape before SMOTE:", X_train7.shape)
print("Test shape before SMOTE:", X_test7.shape)
print("\nTrain target distribution before SMOTE:")
print(y_train7.value_counts())
print("\nTest target distribution:")
print(y_test7.value_counts())


# 8. Apply SMOTE if imbalance exists
ratio = y_train7.value_counts().min() / y_train7.value_counts().max()
if ratio < 0.7:  # imbalance threshold
    smote = SMOTE(random_state=42)
    X_train7, y_train7 = smote.fit_resample(X_train7, y_train7)
    print("\nSMOTE applied to balance classes.")
else:
    print("\nSMOTE not applied (classes already balanced).")

# Reset indices after SMOTE
X_train7 = pd.DataFrame(X_train7, columns=X.columns)
y_train7 = pd.Series(y_train7).reset_index(drop=True)


# 9. Print shapes and distributions after SMOTE
print("\nTrain shape after SMOTE:", X_train7.shape)
print("\nTrain target distribution after SMOTE:")
print(y_train7.value_counts())


# 10. Target meaning
print("\nTarget meaning for Hepatitis dataset:")
print("0 → Live (Normal)")
print("1 → Die (Disease)")


# 11. Show head of training set
print("\nHead of training set:")
print(X_train7.head().assign(target=y_train7.head()))


# 12. Create final train/test DataFrames
train7 = pd.concat([X_train7, y_train7], axis=1)
test7  = pd.concat([X_test7, y_test7.reset_index(drop=True)], axis=1)


Train shape before SMOTE: (124, 19)
Test shape before SMOTE: (31, 19)

Train target distribution before SMOTE:
target
0    98
1    26
Name: count, dtype: int64

Test target distribution:
target
0    25
1     6
Name: count, dtype: int64

SMOTE applied to balance classes.

Train shape after SMOTE: (196, 19)

Train target distribution after SMOTE:
target
0    98
1    98
Name: count, dtype: int64

Target meaning for Hepatitis dataset:
0 → Live (Normal)
1 → Die (Disease)

Head of training set:
        age  sex  steroid  antivirals  fatigue  malaise  anorexia  liver_big  \
0  0.323944  0.0      1.0         1.0      0.0      1.0       1.0        1.0   
1  0.338028  0.0      0.0         1.0      0.0      1.0       1.0        1.0   
2  0.239437  0.0      1.0         1.0      1.0      1.0       1.0        1.0   
3  0.535211  0.0      0.0         1.0      0.0      1.0       1.0        1.0   
4  0.239437  0.0      0.0         1.0      0.0      1.0       1.0        1.0   

   liver_firm  spleen_pal

## 8) parkinsons

In [9]:
# Load dataset
file_path = "parkinsons.data"
df = pd.read_csv(file_path)

# Identify features and target
if "name" in df.columns:
    X = df.drop(["name", "status"], axis=1)
else:
    X = df.drop(["status"], axis=1)

y = df["status"]

# Standardize target
# Original: 0 = Healthy, 1 = Parkinson’s
# Standard form: 0 = Normal/Live, 1 = Disease
y = y.apply(lambda val: 0 if val == 0 else 1)

# Handle missing values in features (numeric → mean)
X = X.fillna(X.mean())

# Train/Test split (stratified to maintain target ratio)
X_train8, X_test8, y_train8, y_test8 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features using MinMaxScaler
scaler = MinMaxScaler()
X_train8 = pd.DataFrame(scaler.fit_transform(X_train8), columns=X.columns)
X_test8 = pd.DataFrame(scaler.transform(X_test8), columns=X.columns)

# Check imbalance in training set
ratio = y_train8.value_counts().min() / y_train8.value_counts().max()
if ratio < 0.7:  # Threshold for imbalance
    smote = SMOTE(random_state=42)
    X_train8, y_train8 = smote.fit_resample(X_train8, y_train8)
    print("\nApplied SMOTE to balance the training set")
else:
    print("\nSMOTE not applied (training set already balanced)")

# Verify shapes and distributions
print("\nTrain shape after SMOTE:", X_train8.shape)
print("Test shape:", X_test8.shape)
print("\nTrain target distribution after SMOTE:")
print(y_train8.value_counts())
print("\nTest target distribution:")
print(y_test8.value_counts())

# Target meaning
print("\nTarget meaning for Parkinson's dataset:")
print("0 → Normal / Live")
print("1 → Disease (Parkinson’s present)")

# Create train8/test8 DataFrames
train8 = pd.DataFrame(X_train8, columns=X_train8.columns)
train8["target"] = y_train8.reset_index(drop=True)

test8 = pd.DataFrame(X_test8, columns=X_test8.columns)
test8["target"] = y_test8.reset_index(drop=True)

# Preview training set
print("\nHead of training set:")
print(train8.head())



Applied SMOTE to balance the training set

Train shape after SMOTE: (236, 22)
Test shape: (39, 22)

Train target distribution after SMOTE:
status
0    118
1    118
Name: count, dtype: int64

Test target distribution:
status
1    29
0    10
Name: count, dtype: int64

Target meaning for Parkinson's dataset:
0 → Normal / Live
1 → Disease (Parkinson’s present)

Head of training set:
   MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  \
0     0.130440      0.023377      0.199564        0.059720          0.090909   
1     0.515363      0.211101      0.056813        0.183926          0.130435   
2     0.128775      0.051984      0.230739        0.103558          0.130435   
3     0.866806      0.296357      0.919727        0.041296          0.011858   
4     0.666418      0.264068      0.119498        0.064168          0.051383   

   MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer(dB)  ...  \
0  0.049133  0.057878    0.049125      0.048384          0.039

## 9) Pima Indians Diabetes Database

In [10]:
# 1. Load dataset
file_path = 'diabetes.csv'
df = pd.read_csv(file_path)
print("Original Shape:", df.shape)

# 2. Define features and target
X = df.drop(['Outcome'], axis=1)
y = df['Outcome']

# 3. Replace invalid zeros with NaN in selected columns
columns_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X[columns_with_zeros] = X[columns_with_zeros].replace(0, np.nan)

# 4. Fill missing values with column mean
X.fillna(X.mean(), inplace=True)

# 5. Verify no NaNs remain
assert not X.isna().any().any(), "Features still contain NaN!"
assert not y.isna().any(), "Target contains NaN!"

# 6. Train/Test split (stratified)
X_train9, X_test9, y_train9, y_test9 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Scale features to 0–1 range
scaler = MinMaxScaler()
X_train9 = pd.DataFrame(scaler.fit_transform(X_train9), columns=X.columns)
X_test9 = pd.DataFrame(scaler.transform(X_test9), columns=X.columns)

# 8. Create final train/test DataFrames
train9 = X_train9.copy()
train9["target"] = y_train9.reset_index(drop=True)

test9 = X_test9.copy()
test9["target"] = y_test9.reset_index(drop=True)

# 9. Print summary and preview
print("\nTrain shape:", train9.shape)
print("Test shape:", test9.shape)

print("\nTrain target distribution:\n", y_train9.value_counts())
print("\nTest target distribution:\n", y_test9.value_counts())

print("\nTarget meaning for Pima Indian Diabetes dataset:")
print("0 → No Diabetes")
print("1 → Diabetes")

print("\nHead of training set:")
print(train9.head())


Original Shape: (768, 9)

Train shape: (614, 9)
Test shape: (154, 9)

Train target distribution:
 Outcome
0    400
1    214
Name: count, dtype: int64

Test target distribution:
 Outcome
0    100
1     54
Name: count, dtype: int64

Target meaning for Pima Indian Diabetes dataset:
0 → No Diabetes
1 → Diabetes

Head of training set:
   Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0     0.058824  0.237762       0.387755       0.054348  0.038409  0.184049   
1     0.294118  0.489510       0.551020       0.217391  0.009602  0.233129   
2     0.117647  0.342657       0.346939       0.358696  0.108368  0.341513   
3     0.058824  0.629371       0.326531       0.240798  0.192796  0.235174   
4     0.000000  0.272727       0.408163       0.347826  0.123457  0.539877   

   DiabetesPedigreeFunction       Age  target  
0                  0.220935  0.050000       0  
1                  0.158129  0.316667       0  
2                  0.062806  0.066667       0  
3      

## 10) liver disorders

In [11]:
# 1. Load dataset
file_path = 'bupa.data'
columns = ['mcv', 'alkphos', 'sgpt', 'sgot', 'gammagt', 'drinks', 'target']
df = pd.read_csv(file_path, header=None, names=columns)
print("Original Shape:", df.shape)

# 2. Define features and target
X = df.drop(['target'], axis=1)
y = df['target']

# 3. Standardize target
# Original: 1 = liver disorder, 2 = normal → 0 = Normal, 1 = Disease
y = y.map({2: 0, 1: 1})

# 4. Handle missing values
X = X.fillna(X.mean())

# 5. Verify no NaNs remain
assert not X.isna().any().any(), "Features still contain NaN!"
assert not y.isna().any(), "Target contains NaN!"

# 6. Train/Test split (stratified)
X_train10, X_test10, y_train10, y_test10 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Scale features to 0–1
scaler = MinMaxScaler()
X_train10 = pd.DataFrame(scaler.fit_transform(X_train10), columns=X.columns)
X_test10 = pd.DataFrame(scaler.transform(X_test10), columns=X.columns)

# 8. Create final train/test DataFrames
train10 = X_train10.copy()
train10["target"] = y_train10.reset_index(drop=True)

test10 = X_test10.copy()
test10["target"] = y_test10.reset_index(drop=True)

# 9. Print summary and preview
print("\nTrain shape:", train10.shape)
print("Test shape:", test10.shape)

print("\nTrain target distribution:\n", y_train10.value_counts())
print("\nTest target distribution:\n", y_test10.value_counts())

print("\nTarget meaning for BUPA Liver dataset:")
print("0 → Normal / No Liver Disease")
print("1 → Disease / Liver Disorder")

print("\nHead of training set:")
print(train10.head())


Original Shape: (345, 7)

Train shape: (276, 7)
Test shape: (69, 7)

Train target distribution:
 target
0    160
1    116
Name: count, dtype: int64

Test target distribution:
 target
0    40
1    29
Name: count, dtype: int64

Target meaning for BUPA Liver dataset:
0 → Normal / No Liver Disease
1 → Disease / Liver Disorder

Head of training set:
        mcv   alkphos      sgpt      sgot   gammagt  drinks  target
0  0.710526  0.739130  0.324503  0.337838  0.304795   0.600       0
1  0.736842  0.556522  0.092715  0.121622  0.071918   0.100       1
2  0.763158  0.365217  0.119205  0.135135  0.020548   0.025       1
3  0.710526  0.608696  0.357616  0.364865  0.393836   0.350       1
4  0.605263  0.617391  0.145695  0.135135  0.010274   0.025       1


## 11) Cardiovascular Disease dataset

In [12]:
# 1. Load dataset
file_path = 'cardio_train.csv'  
cardio = pd.read_csv(file_path, sep=';')
print("Original Shape:", cardio.shape)

# 2. Define features and target
target_column = 'cardio'
X = cardio.drop(columns=[target_column])
y = cardio[target_column]

# 3. Handle missing values
numeric_cols = X.select_dtypes(include='number').columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].mean())

# Encode categorical variables (if any)
categorical_cols = X.select_dtypes(include='object').columns
if len(categorical_cols) > 0:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 4. Verify no NaNs remain
assert not X.isna().any().any(), "Features still contain NaN!"
assert not y.isna().any(), "Target contains NaN!"

# 5. Train/Test split 
X_train11, X_test11, y_train11, y_test11 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Scale features to 0–1
scaler = MinMaxScaler()
X_train11 = pd.DataFrame(scaler.fit_transform(X_train11), columns=X.columns)
X_test11 = pd.DataFrame(scaler.transform(X_test11), columns=X.columns)

# 7. Create train/test DataFrames
train11 = X_train11.copy()
train11["target"] = y_train11.reset_index(drop=True)

test11 = X_test11.copy()
test11["target"] = y_test11.reset_index(drop=True)

# 8. Print summary and preview
print("\nTrain shape:", train11.shape)
print("Test shape:", test11.shape)

print("\nTrain target distribution:\n", y_train11.value_counts())
print("\nTest target distribution:\n", y_test11.value_counts())

print("\nTarget meaning for Cardiovascular dataset:")
print("0 → Normal / No Disease")
print("1 → Disease / Cardiovascular present")

print("\nHead of training set:")
print(train11.head())


Original Shape: (70000, 13)

Train shape: (56000, 13)
Test shape: (14000, 13)

Train target distribution:
 cardio
0    28017
1    27983
Name: count, dtype: int64

Test target distribution:
 cardio
0    7004
1    6996
Name: count, dtype: int64

Target meaning for Cardiovascular dataset:
0 → Normal / No Disease
1 → Disease / Cardiovascular present

Head of training set:
         id       age  gender    height    weight     ap_hi     ap_lo  \
0  0.833278  0.633990     1.0  0.544041  0.380952  0.016698  0.013550   
1  0.861969  0.503390     0.0  0.523316  0.280423  0.016698  0.013550   
2  0.591586  0.635705     0.0  0.559585  0.444444  0.019171  0.015357   
3  0.163992  0.742539     0.0  0.554404  0.380952  0.018553  0.015357   
4  0.294703  0.574846     0.0  0.512953  0.216931  0.015461  0.012376   

   cholesterol  gluc  smoke  alco  active  target  
0          0.0   0.0    0.0   0.0     0.0       1  
1          0.0   0.0    0.0   0.0     1.0       0  
2          0.5   0.0    0.0   0.0 

## Dataset summary table

In [13]:
import pandas as pd

data = [
    [1, "Heart Disease (Cleveland)", "Medical", "(303, 13)", 13, "(242, 14)", "(61, 14)", "0:164, 1:139", "0 = No Disease, 1 = Disease"],
    [2, "Statlog Heart Disease", "Medical", "(270, 13)", 13, "(216, 14)", "(54, 14)", "0:120, 1:150", "0 = No Disease, 1 = Disease"],
    [3, "SPECT Heart", "Medical", "(267, 23)", 22, "(213, 23)", "(54, 23)", "0:55, 1:212", "0 = Normal, 1 = Abnormal"],
    [4, "SPECTF Heart", "Medical", "(267, 45)", 44, "(213, 45)", "(54, 45)", "0:44, 1:169", "0 = Normal, 1 = Disease"],
    [5, "EEG Eye State", "Biomedical", "(14980, 15)", 14, "(11984, 15)", "(2996, 15)", "0:6606, 1:5378", "0 = Eyes Open, 1 = Eyes Closed"],
    [6, "Breast Cancer Wisconsin (WDBC)", "Medical", "(569, 30)", 30, "(455, 31)", "(114, 31)", "0:285, 1:170", "0 = Benign, 1 = Malignant"],
    [7, "Hepatitis (SMOTE)", "Medical", "(155, 19)", 18, "(196, 19)", "(31, 19)", "0:98, 1:98", "0 = Live, 1 = Die"],
    [8, "Parkinson’s Disease (SMOTE)", "Medical", "(195, 22)", 22, "(236, 22)", "(39, 22)", "0:118, 1:118", "0 = Normal, 1 = Disease"],
    [9, "Pima Indians Diabetes", "Medical", "(768, 9)", 8, "(614, 9)", "(154, 9)", "0:400, 1:214", "0 = No Diabetes, 1 = Diabetes"],
    [10, "Liver Disorders (BUPA)", "Medical", "(345, 7)", 6, "(276, 7)", "(69, 7)", "0:160, 1:116", "0 = Normal, 1 = Disease"],
    [11, "Cardiovascular Disease", "Medical", "(70000, 13)", 12, "(56000, 13)", "(14000, 13)", "0:28017, 1:27983", "0 = Normal, 1 = Disease"]
]

columns = [
    "No", "Dataset Name", "Domain", "Original Shape",
    "No. of Features", "Train Shape", "Test Shape",
    "Class Distribution (Train)", "Target Meaning"
]

df = pd.DataFrame(data, columns=columns)

#dataset 7 & 8 smote applied
df 


,No,Dataset Name,Domain,Original Shape,No. of Features,Train Shape,Test Shape,Class Distribution (Train),Target Meaning
0,1,Heart Disease (Cleveland),Medical,"(303, 13)",13,"(242, 14)","(61, 14)","0:164, 1:139","0 = No Disease, 1 = Disease"
1,2,Statlog Heart Disease,Medical,"(270, 13)",13,"(216, 14)","(54, 14)","0:120, 1:150","0 = No Disease, 1 = Disease"
2,3,SPECT Heart,Medical,"(267, 23)",22,"(213, 23)","(54, 23)","0:55, 1:212","0 = Normal, 1 = Abnormal"
3,4,SPECTF Heart,Medical,"(267, 45)",44,"(213, 45)","(54, 45)","0:44, 1:169","0 = Normal, 1 = Disease"
4,5,EEG Eye State,Biomedical,"(14980, 15)",14,"(11984, 15)","(2996, 15)","0:6606, 1:5378","0 = Eyes Open, 1 = Eyes Closed"
5,6,Breast Cancer Wisconsin (WDBC),Medical,"(569, 30)",30,"(455, 31)","(114, 31)","0:285, 1:170","0 = Benign, 1 = Malignant"
6,7,Hepatitis (SMOTE),Medical,"(155, 19)",18,"(196, 19)","(31, 19)","0:98, 1:98","0 = Live, 1 = Die"
7,8,Parkinson’s Disease (SMOTE),Medical,"(195, 22)",22,"(236, 22)","(39, 22)","0:118, 1:118","0 = Normal, 1 = Disease"
8,9,Pima Indians Diabetes,Medical,"(768, 9)",8,"(614, 9)","(154, 9)","0:400, 1:214","0 = No Diabetes, 1 = Diabetes"
9,10,Liver Disorders (BUPA),Medical,"(345, 7)",6,"(276, 7)","(69, 7)","0:160, 1:116","0 = Normal, 1 = Disease"


## Implementation of mlp, xgboost

In [14]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.exceptions import ConvergenceWarning
import warnings
from xgboost import XGBClassifier

# Suppress MLP convergence warnings 
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Dataset names 
dataset_names = {
    1: "Cleveland",
    2: "Statlog",
    3: "SPECT",
    4: "SPECTF",
    5: "EEG Eye State (Eric)",   
    6: "WBC",
    7: "Hepatitis",
    8: "Parkinson",
    9: "Pima Indian Diabetes",
    10: "BUPA Liver Disease",
    11: "Cardiovascular Dataset"
}

# Loop through all datasets
for i in range(1, 12):  # 1 to 11
    dataset_name = dataset_names[i]
    train_key, test_key = f"train{i}", f"test{i}"
    
    # Skip 
    if train_key not in globals() or test_key not in globals():
        print(f"\nSkipping {dataset_name} (missing {train_key}/{test_key})")
        continue

    # Load train/test data
    train = globals()[train_key]
    test  = globals()[test_key]
    
    # Separate features and target
    X_train = train.drop('target', axis=1)
    y_train = train['target']
    X_test  = test.drop('target', axis=1)
    y_test  = test['target']
    
    # Standardize features 
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    
    print(f"\n==============================")
    print(f"Dataset: {dataset_name}")
    print(f"==============================")
    
    # ----- MLP Classifier ----
    mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)
    mlp.fit(X_train_scaled, y_train)
    y_pred_mlp = mlp.predict(X_test_scaled)
    
    print("MLP Classifier:")
    print(f"Accuracy   : {accuracy_score(y_test, y_pred_mlp)*100:.2f}%")
    print(f"Precision  : {precision_score(y_test, y_pred_mlp, average='weighted')*100:.2f}%")
    print(f"Recall     : {recall_score(y_test, y_pred_mlp, average='weighted')*100:.2f}%")
    print(f"F1 Score   : {f1_score(y_test, y_pred_mlp, average='weighted')*100:.2f}%")
    
    # ----- XGBoost Classifier -----
    xgb = XGBClassifier(
        n_estimators=200,       # number of trees
        learning_rate=0.05,     # smaller learning rate for stability
        max_depth=5,            # tree depth
        random_state=42,
        eval_metric='mlogloss'  # avoids warnings for classification
    )
    xgb.fit(X_train, y_train)  
    y_pred_xgb = xgb.predict(X_test)
    
    print("\nXGBoost Classifier:")
    print(f"Accuracy   : {accuracy_score(y_test, y_pred_xgb)*100:.2f}%")
    print(f"Precision  : {precision_score(y_test, y_pred_xgb, average='weighted')*100:.2f}%")
    print(f"Recall     : {recall_score(y_test, y_pred_xgb, average='weighted')*100:.2f}%")
    print(f"F1 Score   : {f1_score(y_test, y_pred_xgb, average='weighted')*100:.2f}%")
    print("------------------------------")



Dataset: Cleveland
MLP Classifier:
Accuracy   : 85.25%
Precision  : 87.43%
Recall     : 85.25%
F1 Score   : 85.19%

XGBoost Classifier:
Accuracy   : 86.89%
Precision  : 87.66%
Recall     : 86.89%
F1 Score   : 86.90%
------------------------------

Dataset: Statlog
MLP Classifier:
Accuracy   : 74.07%
Precision  : 74.42%
Recall     : 74.07%
F1 Score   : 74.15%

XGBoost Classifier:
Accuracy   : 70.37%
Precision  : 71.31%
Recall     : 70.37%
F1 Score   : 70.45%
------------------------------

Dataset: SPECT
MLP Classifier:
Accuracy   : 85.19%
Precision  : 84.55%
Recall     : 85.19%
F1 Score   : 82.89%

XGBoost Classifier:
Accuracy   : 87.04%
Precision  : 86.65%
Recall     : 87.04%
F1 Score   : 85.49%
------------------------------

Dataset: SPECTF
MLP Classifier:
Accuracy   : 81.48%
Precision  : 82.83%
Recall     : 81.48%
F1 Score   : 82.03%

XGBoost Classifier:
Accuracy   : 74.07%
Precision  : 74.07%
Recall     : 74.07%
F1 Score   : 74.07%
------------------------------

Dataset: EEG Eye

## Nsga-III Implementation

* NSGA-III optimizes MLP hyperparameters.
* Hidden: 10–200 | Alpha: 0.0001–1.0 | LR: 0.0001–0.1.
* Pop: 91 | Gen: 20.
* Goals: ↑ Accuracy, ↓ Log Loss, ↓ Complexity.
* MLP max_iter = 1000

In [13]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.util.ref_dirs import get_reference_directions
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Dataset names
dataset_names = {
    1: "Cleveland",
    2: "Statlog",
    3: "SPECT",
    4: "SPECTF",
    5: "EEG Eye State (Eric)",
    6: "WBC",
    7: "Hepatitis",
    8: "Thyroid Disease",
    9: "Parkinson",
    10: "Pima Indian Diabetes",
    11: "BUPA Liver Disease"
}

for i in dataset_names.keys():

    dataset_name = dataset_names[i]
    train_key, test_key = f"train{i}", f"test{i}"

    if train_key not in globals() or test_key not in globals():
        print(f"\nSkipping {dataset_name} (missing {train_key}/{test_key})")
        continue

    train = globals()[train_key]
    test  = globals()[test_key]

    X_train_full = train.drop('target', axis=1)
    y_train_full = train['target']
    X_test  = test.drop('target', axis=1)
    y_test  = test['target']

    # Train-validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        random_state=42,
        stratify=y_train_full
    )

    # Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    print(f"\n==============================")
    print(f"Dataset: {dataset_name}")
    print(f"==============================")

    # ---------- NSGA-III Optimization Problem ----------
    class MLPHyperparamProblem(ElementwiseProblem):

        def __init__(self):
            super().__init__(
                n_var=3,
                n_obj=3,
                xl=np.array([10, 0.0001, 0.0001]),
                xu=np.array([200, 1.0, 0.1])   # LR upper bound FIXED
            )

        def _evaluate(self, x, out, *args, **kwargs):

            hidden_layer_size = int(x[0])
            alpha = float(x[1])
            lr_init = float(x[2])

            mlp = MLPClassifier(
                hidden_layer_sizes=(hidden_layer_size,),
                alpha=alpha,
                learning_rate_init=lr_init,
                max_iter=1000,                # FIXED
                n_iter_no_change=20,          # ADDED (early stopping)
                tol=1e-4,                     # ADDED
                random_state=42
            )

            mlp.fit(X_train_scaled, y_train)

            # Validation loss
            y_val_proba = mlp.predict_proba(X_val_scaled)
            val_loss = log_loss(y_val, y_val_proba)

            # Test accuracy
            y_test_pred = mlp.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_test_pred)

            # Model complexity (approximate)
            n_features = X_train.shape[1]
            n_classes = len(np.unique(y_train))
            model_params = (
                (n_features * hidden_layer_size) +
                hidden_layer_size +
                (hidden_layer_size * n_classes) +
                n_classes
            )

            # NSGA-III minimizes objectives
            out["F"] = [-acc, val_loss, model_params]

    # Reference directions (Das–Dennis)
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)

    algorithm = NSGA3(
        pop_size=len(ref_dirs),
        ref_dirs=ref_dirs
    )

    termination = get_termination("n_gen", 20)

    res = minimize(
        MLPHyperparamProblem(),
        algorithm,
        termination,
        seed=42,
        verbose=True
    )

    # ---------- PRINT RESULTS ----------
    print("\nPareto-optimal solutions:")
    for j, sol in enumerate(res.X):

        hidden_size = int(sol[0])
        alpha_val = sol[1]
        lr_val = sol[2]

        print(
            f"Sol {j+1}: hidden={hidden_size}, "
            f"alpha={alpha_val:.5f}, "
            f"lr={lr_val:.5f}, "
            f"Acc={-res.F[j,0]:.4f}, "
            f"ValLoss={res.F[j,1]:.4f}, "
            f"Params={res.F[j,2]:.0f}"
        )



Dataset: Cleveland
n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       91 |      6 |             - |             -
     2 |      182 |      6 |  0.2712309077 |         nadir
     3 |      273 |      1 |  1.600000E+01 |         ideal
     4 |      364 |      2 |  1.0000000000 |         ideal
     5 |      455 |      2 |  0.000000E+00 |             f
     6 |      546 |      2 |  0.1428571429 |         ideal
     7 |      637 |      2 |  0.4982227641 |         ideal
     8 |      728 |      5 |  0.0831445860 |         ideal
     9 |      819 |      4 |  0.2552016975 |         ideal
    10 |      910 |      4 |  0.0284250619 |         ideal
    11 |     1001 |      6 |  0.2500000000 |         ideal
    12 |     1092 |     10 |  0.0967096008 |             f
    13 |     1183 |      9 |  0.4000000000 |         ideal
    14 |     1274 |     10 |  0.1575845125 |         ideal
    15 |     1365 |     11 |  0.1111111111 |         ideal
    16 |     1456 |     11 |  0.0113

## NSGA-3 with apative fitness and penality

In [14]:
# ============================================================
# ADAPTIVE NSGA-III FRAMEWORK (FINAL FIXED VERSION)
# HV + IGD + Pareto Storage + Adaptive Penalty
# ============================================================

import numpy as np
import os, json, warnings

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, accuracy_score, confusion_matrix

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.optimize import minimize
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.core.callback import Callback
from pymoo.termination import get_termination

# NEW pymoo indicators API
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# ADAPTIVE STATE
# ============================================================

class AdaptiveState:

    def __init__(self, theta_min=0.0, alpha=1.2, beta=1.0, gamma=2.0):
        self.t = 0
        self.r = 0.5
        self.theta_min = theta_min
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def compute_theta(self, T, w_sens=1.0):

        progress = self.t / max(1, T)

        theta = self.theta_min + \
                (progress ** self.alpha) * \
                ((1 - self.r) ** self.beta) * \
                w_sens

        return theta


# ============================================================
# MODEL EVALUATION
# ============================================================

def evaluate_model(x, X_train, y_train, X_val, y_val):

    n_hidden = int(np.clip(np.round(x[0]), 1, 200))
    alpha = 10 ** x[1]
    lr_init = 10 ** x[2]

    clf = MLPClassifier(
        hidden_layer_sizes=(n_hidden,),
        alpha=alpha,
        learning_rate_init=lr_init,
        max_iter=500,          # improved convergence
        early_stopping=True,
        random_state=RANDOM_SEED
    )

    try:
        clf.fit(X_train, y_train)
    except:
        return 0, 1e6, 1e6, 0, len(y_train)

    # Predictions
    y_train_pred = clf.predict(X_train)
    y_val_pred   = clf.predict(X_val)
    y_val_prob   = clf.predict_proba(X_val)

    train_acc = accuracy_score(y_train, y_train_pred)
    val_loss  = log_loss(y_val, y_val_prob)

    # Parameter count
    num_params = sum(
        c.size + b.size
        for c, b in zip(clf.coefs_, clf.intercepts_)
    )

    # FN sensitivity (validation-based)
    tn, fp, fn, tp = confusion_matrix(
        y_val, y_val_pred
    ).ravel()

    return train_acc, val_loss, num_params, tp, fn


# ============================================================
# OPTIMIZATION PROBLEM
# ============================================================

class AdaptiveProblem(ElementwiseProblem):

    def __init__(self, adaptive_state,
                 X_train, y_train, X_val, y_val):

        super().__init__(
            n_var=3,
            n_obj=3,
            n_constr=3,
            xl=np.array([1, -6, -6]),
            xu=np.array([200, -1, -1])
        )

        self.adap = adaptive_state
        self.X_train = X_train
        self.y_train = y_train
        self.X_val   = X_val
        self.y_val   = y_val

        # Constraints
        self.max_params = 700
        self.max_val_loss = 0.4
        self.min_train_acc = 0.8

    def _evaluate(self, x, out, *args, **kwargs):

        train_acc, val_loss, params, TP, FN = evaluate_model(
            x, self.X_train, self.y_train,
            self.X_val, self.y_val
        )

        # Objectives
        f1 = 1 - train_acc
        f2 = val_loss
        f3 = params

        # Constraints
        g1 = params - self.max_params
        g2 = val_loss - self.max_val_loss
        g3 = self.min_train_acc - train_acc

        P = max(0, g1) + max(0, g2) + max(0, g3)

        # FN sensitivity
        denom = TP + FN if TP + FN > 0 else 1
        fn_rate = FN / denom
        w_sens = 1 + self.adap.gamma * fn_rate

        theta = self.adap.compute_theta(
            T=self.adap.T_max,
            w_sens=w_sens
        )

        out["F"] = [
            f1 + theta * P,
            f2 + theta * P,
            f3 + theta * P
        ]

        out["G"] = [g1, g2, g3]


# ============================================================
# CALLBACK → HV + PF STORAGE
# ============================================================

class MetricsCallback(Callback):

    def __init__(self, adaptive_state, ref_point):
        super().__init__()

        self.adap = adaptive_state
        self.hv   = HV(ref_point=ref_point)

        self.data["HV"] = []
        self.data["PF"] = []

    def notify(self, algorithm):

        F = algorithm.pop.get("F")

        # Adaptive update
        cv = np.sum(
            np.maximum(0, algorithm.pop.get("G")),
            axis=1
        )

        feasible = np.sum(cv <= 0)
        self.adap.r = feasible / len(cv)
        self.adap.t += 1

        # Store PF
        self.data["PF"].append(F)

        # Hypervolume
        hv = self.hv(F)
        self.data["HV"].append(hv)


# ============================================================
# RUN NSGA-III
# ============================================================

def run_nsga3(dataset_name,
              X_train, y_train,
              X_val, y_val):

    adap = AdaptiveState()
    adap.T_max = 20

    problem = AdaptiveProblem(
        adap, X_train, y_train, X_val, y_val
    )

    ref_dirs = get_reference_directions(
        "das-dennis", 3, n_partitions=12
    )

    algorithm = NSGA3(
        pop_size=len(ref_dirs),
        ref_dirs=ref_dirs
    )

    ref_point = np.array([1.5, 1.5, 2000])

    callback = MetricsCallback(adap, ref_point)

    res = minimize(
        problem,
        algorithm,
        get_termination("n_gen", 20),
        seed=42,
        callback=callback,
        verbose=True
    )

    # ========================================================
    # IGD COMPUTATION (FINAL PF REFERENCE — CORRECT)
    # ========================================================

    pf_reference = res.F

    igd_metric = IGD(pf_reference)

    igd_per_gen = [
        igd_metric(F) for F in callback.data["PF"]
    ]

    # ========================================================
    # STORE RESULTS1
    # ========================================================

    os.makedirs("results1", exist_ok=True)

    results1 = {
        "dataset": dataset_name,
        "HV_per_gen": callback.data["HV"],
        "IGD_per_gen": igd_per_gen,
        "Pareto_front": res.F.tolist(),
        "Solutions": res.X.tolist()
    }

    with open(f"results1/{dataset_name}.json", "w") as f:
        json.dump(results1, f, indent=4)

    return res


In [15]:
# ============================================================
# RUN ADAPTIVE NSGA-III FOR ALL DATASETS
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Dataset names
dataset_names = {
    1: "Cleveland",
    2: "Statlog",
    3: "SPECT",
    4: "SPECTF",
    5: "EEG Eye State",
    6: "WBC",
    7: "Hepatitis",
    8: "Thyroid",
    9: "Parkinson",
    10: "Pima",
    11: "BUPA"
}

for i in dataset_names.keys():

    dataset_name = dataset_names[i]
    train_key = f"train{i}"
    test_key  = f"test{i}"

    # ---------------------------------------
    # Check dataset availability
    # ---------------------------------------
    if train_key not in globals() or test_key not in globals():
        print(f"\nSkipping {dataset_name} (missing {train_key}/{test_key})")
        continue

    print(f"\n==============================")
    print(f"Running → {dataset_name}")
    print(f"==============================")

    train = globals()[train_key]
    test  = globals()[test_key]

    # ---------------------------------------
    # Split features / target
    # ---------------------------------------
    X_train_full = train.drop("target", axis=1)
    y_train_full = train["target"]

    X_test = test.drop("target", axis=1)
    y_test = test["target"]

    # ---------------------------------------
    # Train / Validation split
    # ---------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        stratify=y_train_full,
        random_state=42
    )

    # ---------------------------------------
    # Feature scaling
    # ---------------------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    print("Train shape :", X_train_scaled.shape)
    print("Val shape   :", X_val_scaled.shape)

    # ---------------------------------------
    # Run Adaptive NSGA-III
    # ---------------------------------------
    res = run_nsga3(
        dataset_name,
        X_train_scaled, y_train,
        X_val_scaled,   y_val
    )

    print(f"{dataset_name} optimization completed ✔")



Running → Cleveland
Train shape : (193, 13)
Val shape   : (49, 13)
n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |       91 |      2 |  0.000000E+00 |  8.404381E+02 |             - |             -
     2 |      182 |      2 |  0.000000E+00 |  9.054846E+01 |  1.6250000000 |         ideal
     3 |      273 |      2 |  0.000000E+00 |  0.2841676850 |  0.4000000000 |         ideal
     4 |      364 |      5 |  0.000000E+00 |  0.0224103230 |  0.0833333333 |         ideal
     5 |      455 |      7 |  0.000000E+00 |  0.000000E+00 |  0.1428571429 |         ideal
     6 |      546 |      9 |  0.000000E+00 |  0.000000E+00 |  0.1250000000 |         ideal
     7 |      637 |      7 |  0.000000E+00 |  0.000000E+00 |  0.0176823789 |         nadir
     8 |      728 |      9 |  0.000000E+00 |  0.000000E+00 |  0.3992921519 |         ideal
     9 |      819 |     10 |  0.000000E+00 |  0.000000E+00 |  0.2498219845 |         ideal
    10 |      910 |   

AttributeError: 'NoneType' object has no attribute 'ndim'

In [13]:
# ============================================================
# ADAPTIVE NSGA-III — PIMA + BUPA (SEPARATE FINAL CODE)
# HV + IGD + Safe PF + Dataset-Specific Constraints
# ============================================================

import numpy as np
import os, json, warnings

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, accuracy_score, confusion_matrix

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.optimize import minimize
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.core.callback import Callback
from pymoo.termination import get_termination

from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# ADAPTIVE STATE
# ============================================================

class AdaptiveState:

    def __init__(self, theta_min=0.0, alpha=1.2, beta=1.0, gamma=2.0):
        self.t = 0
        self.r = 0.5
        self.theta_min = theta_min
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def compute_theta(self, T, w_sens=1.0):

        progress = self.t / max(1, T)

        theta = self.theta_min + \
                (progress ** self.alpha) * \
                ((1 - self.r) ** self.beta) * \
                w_sens

        return theta


# ============================================================
# MODEL EVALUATION
# ============================================================

def evaluate_model(x, X_train, y_train, X_val, y_val):

    n_hidden = int(np.clip(np.round(x[0]), 1, 200))
    alpha    = 10 ** x[1]
    lr_init  = 10 ** x[2]

    clf = MLPClassifier(
        hidden_layer_sizes=(n_hidden,),
        alpha=alpha,
        learning_rate_init=lr_init,
        max_iter=500,
        early_stopping=True,
        random_state=RANDOM_SEED
    )

    try:
        clf.fit(X_train, y_train)
    except:
        return 0, 1e6, 1e6, 0, len(y_train)

    y_train_pred = clf.predict(X_train)
    y_val_pred   = clf.predict(X_val)
    y_val_prob   = clf.predict_proba(X_val)

    train_acc = accuracy_score(y_train, y_train_pred)
    val_loss  = log_loss(y_val, y_val_prob)

    params = sum(
        c.size + b.size
        for c, b in zip(clf.coefs_, clf.intercepts_)
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_val_pred
    ).ravel()

    return train_acc, val_loss, params, tp, fn


# ============================================================
# OPTIMIZATION PROBLEM
# ============================================================

class AdaptiveProblem(ElementwiseProblem):

    def __init__(self, adap,
                 X_train, y_train,
                 X_val, y_val,
                 dataset_name):

        super().__init__(
            n_var=3,
            n_obj=3,
            n_constr=3,
            xl=np.array([1, -6, -6]),
            xu=np.array([200, -1, -1])
        )

        self.adap = adap
        self.X_train = X_train
        self.y_train = y_train
        self.X_val   = X_val
        self.y_val   = y_val

        # ====================================================
        # DATASET-SPECIFIC CONSTRAINTS (FIXED)
        # ====================================================

        if dataset_name == "BUPA":

            self.min_train_acc = 0.60
            self.max_val_loss  = 0.60
            self.max_params    = 1200

        elif dataset_name == "Pima":

            self.min_train_acc = 0.70
            self.max_val_loss  = 0.50
            self.max_params    = 900

        else:

            self.min_train_acc = 0.80
            self.max_val_loss  = 0.40
            self.max_params    = 700


    def _evaluate(self, x, out, *args, **kwargs):

        train_acc, val_loss, params, TP, FN = evaluate_model(
            x,
            self.X_train, self.y_train,
            self.X_val,   self.y_val
        )

        # Objectives
        f1 = 1 - train_acc
        f2 = val_loss
        f3 = params

        # Constraints
        g1 = params - self.max_params
        g2 = val_loss - self.max_val_loss
        g3 = self.min_train_acc - train_acc

        P = max(0, g1) + max(0, g2) + max(0, g3)

        denom = TP + FN if TP + FN > 0 else 1
        fn_rate = FN / denom
        w_sens = 1 + self.adap.gamma * fn_rate

        theta = self.adap.compute_theta(
            T=self.adap.T_max,
            w_sens=w_sens
        )

        out["F"] = [
            f1 + theta * P,
            f2 + theta * P,
            f3 + theta * P
        ]

        out["G"] = [g1, g2, g3]


# ============================================================
# CALLBACK — SAFE HV + PF
# ============================================================

class MetricsCallback(Callback):

    def __init__(self, adap, ref_point):

        super().__init__()

        self.adap = adap
        self.hv   = HV(ref_point=ref_point)

        self.data["HV"] = []
        self.data["PF"] = []


    def notify(self, algorithm):

        F = algorithm.pop.get("F")

        cv = np.sum(
            np.maximum(0, algorithm.pop.get("G")),
            axis=1
        )

        feasible = np.sum(cv <= 0)
        self.adap.r = feasible / len(cv)
        self.adap.t += 1

        if F is None or len(F) == 0:
            self.data["PF"].append(None)
            self.data["HV"].append(np.nan)
        else:
            self.data["PF"].append(F.copy())
            self.data["HV"].append(float(self.hv(F)))


# ============================================================
# RUN NSGA-III
# ============================================================

def run_nsga3(dataset_name,
              X_train, y_train,
              X_val,   y_val):

    adap = AdaptiveState()
    adap.T_max = 20

    problem = AdaptiveProblem(
        adap,
        X_train, y_train,
        X_val,   y_val,
        dataset_name
    )

    ref_dirs = get_reference_directions(
        "das-dennis", 3, n_partitions=12
    )

    algorithm = NSGA3(
        pop_size=len(ref_dirs),
        ref_dirs=ref_dirs
    )

    callback = MetricsCallback(
        adap,
        ref_point=np.array([1.5, 1.5, 2000])
    )

    res = minimize(
        problem,
        algorithm,
        get_termination("n_gen", 20),
        seed=42,
        callback=callback,
        verbose=True
    )

    # ================= SAFE IGD =================

    pf_reference = res.F
    igd_per_gen = []

    if pf_reference is not None and len(pf_reference) > 1:

        igd_metric = IGD(pf_reference)

        for F in callback.data["PF"]:
            if F is None:
                igd_per_gen.append(np.nan)
            else:
                igd_per_gen.append(float(igd_metric(F)))

    else:
        igd_per_gen = [np.nan] * len(callback.data["PF"])


    # ================= SAVE =================

    os.makedirs("results1", exist_ok=True)

    results = {
        "dataset": dataset_name,
        "HV_per_gen": callback.data["HV"],
        "IGD_per_gen": igd_per_gen,
        "Pareto_front":
            [] if res.F is None else res.F.tolist(),
        "Solutions":
            [] if res.X is None else res.X.tolist()
    }

    with open(f"results1/{dataset_name}.json", "w") as f:
        json.dump(results, f, indent=4)

    print(f"\n✔ {dataset_name} optimization completed")


# ============================================================
# DATASET LOOP — ONLY 2 DATASETS
# ============================================================

datasets = {
    "Pima": train10,
    "BUPA": train11
}

tests = {
    "Pima": test10,
    "BUPA": test11
}

for name in datasets:

    print(f"\n==============================")
    print(f"Running → {name}")
    print(f"==============================")

    train = datasets[name]
    test  = tests[name]

    X = train.drop("target", axis=1)
    y = train["target"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    run_nsga3(name, X_train, y_train, X_val, y_val)



Running → Pima
n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |       91 |      1 |  0.000000E+00 |  1.364479E+02 |             - |             -
     2 |      182 |      3 |  0.000000E+00 |  0.1992939424 |  1.1666666667 |         ideal
     3 |      273 |      4 |  0.000000E+00 |  0.0479450004 |  0.8713645012 |         ideal
     4 |      364 |      3 |  0.000000E+00 |  0.0208802329 |  0.0224802475 |         ideal
     5 |      455 |      3 |  0.000000E+00 |  0.0044991305 |  0.2857142857 |         ideal
     6 |      546 |      7 |  0.000000E+00 |  0.0001204515 |  0.4327628810 |         ideal
     7 |      637 |      7 |  0.000000E+00 |  0.000000E+00 |  0.000000E+00 |             f
     8 |      728 |      6 |  0.000000E+00 |  0.000000E+00 |  0.1000000000 |         nadir
     9 |      819 |      6 |  0.000000E+00 |  0.000000E+00 |  0.000000E+00 |             f
    10 |      910 |      7 |  0.000000E+00 |  0.000000E+00 |  0.1666666667